### Seattle Fire 911 Exploration Notebook

Use this notebook to explore the data generated by the pipeline in your preferred programming language.

**Note**: This notebook is not executed as part of the pipeline.

In [ ]:
from databricks.connect import DatabricksSession
import matplotlib.pyplot as plt
import pandas as pd
spark = DatabricksSession.builder.getOrCreate()

In [1]:
%sql
SELECT *
From `seattle_data`.`seattle_bronze`.`seattle_fire_911_calls`
LIMIT 20;

,address,type,datetime,latitude,longitude,incident_number
0,747 N 135th St,Aid Response,2026-03-09T08:40:00.000,47.726803,-122.347793,F260032058
1,401 Broadway E,Aid Response,2026-03-09T08:42:00.000,47.621988,-122.320906,F260032059
2,Sb I5 Fr Ravenna,MVI Freeway,2026-03-09T08:48:00.000,47.673155,-122.321518,F260032060
3,3501 Rainier Ave S,Aid Response,2026-03-09T08:52:00.000,47.571853,-122.293193,F260032061
4,507 W Mercer St,Medic Response,2003-11-07T09:30:48.000,47.624608,-122.363505,F030080834
5,4104 Meridian Av N,Medic Response,2003-11-07T09:31:47.000,47.656799,-122.333806,F030080835
6,11650 Beacon Av S,Auto Fire Alarm,2003-11-07T09:43:15.000,47.498464,-122.263967,F030080841
7,1518 Nw 60th St,1RED 1 Unit,2003-11-07T09:50:05.000,47.67236,-122.37644,F030080842
8,8499 Tillicum Rd Sw,Aid Response Yellow,2003-11-07T09:56:10.000,47.527277,-122.388451,F030080843
9,4515 25th Av Ne,Aid Response,2003-11-07T10:00:57.000,47.661508,-122.300905,F030080844


In [2]:
%sql
SELECT 
    SUBSTRING(`datetime`, 1, 7) AS `year_month`, -- Extract the year and month in the format 'YYYY-MM'
    COUNT(*) AS `incident_count`
FROM 
    `seattle_data`.`seattle_bronze`.`seattle_fire_911_calls`
GROUP BY 
    SUBSTRING(`datetime`, 1, 7)
ORDER BY 
    `year_month`;

,year_month,incident_count
0,2003-11,5151
1,2003-12,6452
2,2004-01,6599
3,2004-02,5815
4,2004-03,6427
5,2004-04,6118
6,2004-05,6394
7,2004-06,6839
8,2004-07,7462
9,2004-08,7243


In [15]:
# Plot incident counts by year_month using Spark and matplotlib
df = spark.sql("""
SELECT YEAR(`datetime`) AS year, MONTH(`datetime`) AS month, COUNT(*) AS incident_count
FROM seattle_data.seattle_bronze.seattle_fire_911_calls
GROUP BY year, month
ORDER BY year, month
"""
)

pdf = df.toPandas()

In [16]:
import plotly.graph_objects as go

pdf = pdf.sort_values(['year', 'month']).reset_index(drop=True)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pdf['x_label'] = pdf['year'].astype(str) + '-' + pdf['month'].astype(str).str.zfill(2)

fig = go.Figure()

# Add summer (Jun-Aug) and winter (Dec-Feb) shading
summer_months = [6, 7, 8]
winter_months = [12, 1, 2]

for _, row in pdf.iterrows():
    if row['month'] in summer_months:
        color = 'rgba(255, 200, 0, 0.15)'   # yellow for summer
    elif row['month'] in winter_months:
        color = 'rgba(100, 180, 255, 0.15)'  # blue for winter
    else:
        continue

    idx = row.name
    x0 = pdf.loc[idx, 'x_label']
    x1 = pdf.loc[idx + 1, 'x_label'] if idx + 1 < len(pdf) else x0

    fig.add_vrect(x0=x0, x1=x1, fillcolor=color, layer='below', line_width=0)

fig.add_trace(go.Scatter(
    x=pdf['x_label'],
    y=pdf['incident_count'],
    mode='lines+markers',
    hovertemplate='<b>%{x}</b><br>Incidents: %{y:,}<extra></extra>',
    line=dict(color='steelblue', width=2),
    marker=dict(size=5),
))

# Add legend entries for the shading
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(255, 200, 0, 0.5)', symbol='square'),
    name='Summer (Jun-Aug)'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(100, 180, 255, 0.5)', symbol='square'),
    name='Winter (Dec-Feb)'))

fig.update_layout(
    title='Seattle Fire 911 Incident Counts by Month',
    xaxis=dict(tickangle=45, tickfont=dict(size=10)),
    yaxis=dict(title='Incident Count', tickformat=','),
    height=550,
    hovermode='x unified',
    legend=dict(x=1.01, y=1),
)

fig.show()

In [5]:
%sql
SELECT 
    type AS `Emergency_Type`,
    COUNT(*) AS `Incident_Count`
FROM 
    `seattle_data`.`seattle_bronze`.`seattle_fire_911_calls`
GROUP BY 
    type
ORDER BY 
    `Incident_Count` DESC;  

,Emergency_Type,Incident_Count
0,Aid Response,1043121
1,Medic Response,342146
2,Auto Fire Alarm,122693
3,Trans to AMR,82678
4,Aid Response Yellow,43070
5,Triaged Incident,31822
6,Automatic Fire Alarm Resd,31275
7,Motor Vehicle Accident,28959
8,MVI - Motor Vehicle Incident,28627
9,1RED 1 Unit,24378


In [6]:
# Plot incident counts by emergency type using Spark and matplotlib
df = spark.sql("""
SELECT 
    type AS `Emergency_Type`,
    COUNT(*) AS `Incident_Count`
FROM 
    `seattle_data`.`seattle_bronze`.`seattle_fire_911_calls`
GROUP BY 
    type
ORDER BY 
    `Incident_Count` DESC;  
"""
)

pdf = df.toPandas()

display(pdf)

,Emergency_Type,Incident_Count
0,Aid Response,1043121
1,Medic Response,342146
2,Auto Fire Alarm,122693
3,Trans to AMR,82678
4,Aid Response Yellow,43070
...,...,...
264,4RED - 2 + 1 + 1 Water,1
265,Multiple Casualty Incident Maj,1
266,Mutual Aid- Adv. Life,1
267,Mutual Aid- Strike Team Lad,1


In [7]:
fig = go.Figure([go.Bar(x=pdf['Emergency_Type'], y=pdf['Incident_Count'])])
fig.show()

In [ ]:
import plotly.express as px
import pandas as pd
# Assuming your df has 'latitude' and 'longitude' columns
pdf_agg = spark.sql("""
    SELECT 
        ROUND(CAST(latitude AS DOUBLE), 3) AS lat_r,
        ROUND(CAST(longitude AS DOUBLE), 3) AS lon_r,
        COUNT(*) AS count
    FROM seattle_data.seattle_bronze.seattle_fire_911_calls
    WHERE latitude IS NOT NULL 
      AND longitude IS NOT NULL
      AND CAST(latitude AS DOUBLE) IS NOT NULL
      AND CAST(longitude AS DOUBLE) IS NOT NULL
    GROUP BY 
        ROUND(CAST(latitude AS DOUBLE), 3),
        ROUND(CAST(longitude AS DOUBLE), 3)
""").toPandas()

fig = px.density_mapbox(
    pdf_agg,
    lat='lat_r',
    lon='lon_r',
    z='count',
    radius=10,
    center=dict(lat=47.6062, lon=-122.3321),
    zoom=10,
    mapbox_style='carto-positron',
    title='Seattle Fire 911 Call Density',
    height=700,
)

fig.show()

/tmp/ipykernel_21653/2388153866.py:21: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(
